# ⛰️ 01-数据结构与算法 · 主题 11：堆与 Top-K

> 堆是「动态取最大/最小」与「Top-K / 流式中位数」的利器，Python 用 `heapq` 一行上手。
> 本 notebook 目标：**吃透堆的性质与 heapq → 背熟 Top-K 模板 → 吃透 5 道高频题 → 通过 25 道自测**。

**前置**：主题 3（链表）、主题 4（栈队列）。**运行环境**：Python 3.8+（`heapq`/`collections`）。

> **直觉类比 · 淘汰赛**：堆就像体育**淘汰赛**——每轮只需和当前「最强候选」比一次（父子节点比较），$\log n$ 轮内就能确定冠军，不必与全员比较。
> **Top-K · 守门员**：Top-K 的小顶堆 = 只留一个「**守门员**」（堆顶 = 第 K 大）；新选手先跟守门员比，赢过守门员才换人进场，否则直接淘汰——门将永远保持第 K 大。


## §1 堆的性质与 heapq

**二叉堆** = 完全二叉树 + 堆序性质：

- 小顶堆：$parent \le children$（堆顶最小）
- 大顶堆：$parent \ge children$（堆顶最大）

Python 的 `heapq` 默认**小顶堆**；要大顶堆就把值取负存进去。

```python
import heapq
hp = []
heapq.heappush(hp, 3)          # O(log n)
heapq.heappop(hp)              # O(log n) 弹出最小
hp[0]                          # O(1) 查看最小
heapq.heapify(arr)             # O(n) 原地建堆
heapq.nlargest(k, it)          # 便捷版
```

### 复杂度表（背熟）

| 操作 | 复杂度 |
|------|--------|
| 查看堆顶 | $O(1)$ |
| 插入 / 删除堆顶 | $O(\log n)$ |
| 建堆 heapify | $O(n)$（不是 $O(n \log n)$！） |
| n 次插入 | $O(n \log n)$ |

**为什么 heapify 是 $O(n)$**：从最后一个非叶节点自底向上下沉，第 $h$ 层 $n/2^{h+1}$ 个节点各做 $O(h)$ 下沉：

$$\sum_{h=0}^{\log n} \frac{n}{2^{h+1}} \cdot O(h) = O(n)$$

## §2 手写二叉堆（面试加分）

堆用数组存完全二叉树：下标 $i$ 的左孩子 $2i+1$、右孩子 $2i+2$、父节点 $\lfloor (i-1)/2 \rfloor$。

**两个核心操作**：

- **上浮（sift up）**：插入末尾后与父节点比较交换，直到堆序恢复
- **下沉（sift down）**：删堆顶后把末尾移到堆顶，与较小孩子交换

```python
class MinHeap:
    def __init__(self): self.a = []
    def push(self, x):
        self.a.append(x)
        self._sift_up(len(self.a) - 1)
    def pop(self):
        top = self.a[0]
        self.a[0] = self.a[-1]
        self.a.pop()
        self._sift_down(0)
        return top
```

In [ ]:
class MinHeap:
    def __init__(self):
        self.a = []

    def _sift_up(self, i):
        while i > 0:
            p = (i - 1) // 2
            if self.a[p] <= self.a[i]:
                break
            self.a[p], self.a[i] = self.a[i], self.a[p]
            i = p

    def _sift_down(self, i):
        n = len(self.a)
        while True:
            l, r = 2 * i + 1, 2 * i + 2
            smallest = i
            if l < n and self.a[l] < self.a[smallest]:
                smallest = l
            if r < n and self.a[r] < self.a[smallest]:
                smallest = r
            if smallest == i:
                break
            self.a[i], self.a[smallest] = self.a[smallest], self.a[i]
            i = smallest

    def push(self, x):
        self.a.append(x)
        self._sift_up(len(self.a) - 1)

    def pop(self):
        top = self.a[0]
        last = self.a.pop()
        if self.a:
            self.a[0] = last
            self._sift_down(0)
        return top

h = MinHeap()
for x in [5, 3, 8, 1, 9, 2]:
    h.push(x)
out = []
while h.a:
    out.append(h.pop())
print('手写小顶堆弹出顺序:', out)   # 期望 [1, 2, 3, 5, 8, 9]（升序 = 堆排序）
print('复杂度：push/pop 均 O(log n)，等价堆排序 O(n log n)')

In [ ]:
class MaxHeap:
    """大顶堆：与 MinHeap 结构相同，仅把比较符号翻转（parent >= children）。"""
    def __init__(self):
        self.a = []

    def _sift_up(self, i):
        while i > 0:
            p = (i - 1) // 2
            if self.a[p] >= self.a[i]:          # 父 >= 子：堆序已满足
                break
            self.a[p], self.a[i] = self.a[i], self.a[p]
            i = p

    def _sift_down(self, i):
        n = len(self.a)
        while True:
            l, r = 2 * i + 1, 2 * i + 2
            largest = i
            if l < n and self.a[l] > self.a[largest]:   # 找更大的孩子
                largest = l
            if r < n and self.a[r] > self.a[largest]:
                largest = r
            if largest == i:
                break
            self.a[i], self.a[largest] = self.a[largest], self.a[i]
            i = largest

    def push(self, x):
        self.a.append(x)
        self._sift_up(len(self.a) - 1)

    def pop(self):
        top = self.a[0]
        last = self.a.pop()
        if self.a:
            self.a[0] = last
            self._sift_down(0)
        return top

    def peek(self):
        return self.a[0]


h = MaxHeap()
nums = [3, 1, 4, 1, 5, 9, 2]
for x in nums:
    h.push(x)
out = []
while h.a:
    out.append(h.pop())
print('手写大顶堆弹出顺序:', out)              # 期望 [9, 5, 4, 3, 2, 1, 1]（从大到小）
assert out == sorted(nums, reverse=True), 'MaxHeap 弹出顺序错误'
print('断言通过：大顶堆从大到小弹出 ✔')

h2 = MaxHeap()
h2.push(7); h2.push(10); h2.push(3)
print('peek（堆顶最大）:', h2.peek())          # 10
assert h2.peek() == 10, 'peek 错误'
print('断言通过：peek 返回堆顶最大 ✔')


In [ ]:
from collections import Counter
import heapq


class LazyHeap:
    """小顶堆 + 懒删除：remove(x) 只记账不真删，pop() 前先清掉堆顶已删除元素。"""
    def __init__(self):
        self.hp = []
        self.deleted = Counter()      # 每个值被标记删除的次数
        self.size = 0                 # 逻辑大小（含已标记删除的）

    def push(self, x):
        heapq.heappush(self.hp, x)
        self.size += 1

    def remove(self, x):
        self.deleted[x] += 1          # 不真正删除，先记账
        self.size -= 1

    def _clean(self):
        while self.hp and self.deleted[self.hp[0]] > 0:
            self.deleted[heapq.heappop(self.hp)] -= 1

    def pop(self):
        self._clean()                 # 堆顶若已被删除，先清理掉
        x = heapq.heappop(self.hp)
        self.size -= 1
        return x

    def __len__(self):
        return self.size


h = LazyHeap()
for x in [5, 3, 8, 1]:
    h.push(x)
print('push [5,3,8,1] 后 remove(3)（打标记，不真删）')
h.remove(3)
print('deleted 计数:', dict(h.deleted))
out = []
while h:
    out.append(h.pop())
print('pop 依次:', out)                # 期望 [1, 5, 8]，3 已被过滤
assert out == [1, 5, 8] and 3 not in out, '懒删除语义错误'
print('断言通过：remove 后 pop 不会返回被删元素 ✔')


## §3 Top-K 模板：第 K 大 / 前 K 高频

### 215 数组中的第 K 个最大元素

**思路**：维护**大小为 K 的小顶堆**——堆里始终是「当前已见的前 K 大」，堆顶就是第 K 大：

```python
hp = nums[:k]; heapq.heapify(hp)
for x in nums[k:]:
    if x > hp[0]:
        heapq.heapreplace(hp, x)   # 弹出最小，压入更大
return hp[0]
```

**复杂度**（背）：时间 $O(n \log k)$（每个元素至多一次替换 $O(\log k)$），空间 $O(k)$。

**为什么比全排序好**：$k \ll n$ 时省内存省时间；且支持**流式**（还可动态增删）。

### 347 前 K 个高频元素

**两步**：Counter 统计频次 → 大小为 K 的小顶堆按频次过滤：

$$\text{堆元素} = (\text{freq}, \text{num}), \quad \text{堆顶即当前第 K 高频的频次}$$

### 703 数据流中的第 K 大元素

**思路**：维护一个**大小为 k 的小顶堆**，堆里始终是已见数据中最大的 k 个，**堆顶就是第 K 大**。

新数到达时：堆未满直接 `heappush`；否则若 `x > 堆顶` 就 `heapq.heappushpop(hp, x)`（先压入 x 再弹走当前最小，等价于替换），堆顶保持第 K 大。

单次 `add` 为 $O(\log k)$，空间 $O(k)$，天然支持流式数据。


In [ ]:
import heapq
from collections import Counter

def find_kth_largest(nums, k):
    hp = nums[:k]
    heapq.heapify(hp)                 # 小顶堆，大小为 k
    for x in nums[k:]:
        if x > hp[0]:
            heapq.heapreplace(hp, x)
    return hp[0]

def top_k_frequent(nums, k):
    cnt = Counter(nums)
    hp = []                           # (频率, 数)，小顶堆
    for num, freq in cnt.items():
        heapq.heappush(hp, (freq, num))
        if len(hp) > k:
            heapq.heappop(hp)         # 弹掉当前最小频率
    return [num for _, num in hp]

print('第3大:', find_kth_largest([3, 2, 1, 5, 6, 4], 3))            # 期望 4
print('第1大:', find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 1))   # 期望 6
print('前2高频:', sorted(top_k_frequent([1, 1, 1, 2, 2, 3], 2)))     # 期望 [1, 2]
print('前1高频:', top_k_frequent([1], 1))                           # 期望 [1]
print('复杂度: O(n log k) 时间, O(k) 空间')

## §4 数据流中的中位数（LeetCode 295）：双堆

**题目**：动态插入整数，随时查中位数。

**思路**：两个堆把数据**对半分**：

- 小顶堆 `lo` 存**较小的一半**（用大顶堆实现，即存负数）——堆顶是较小半的最大值
- 大顶堆 `hi` 存较大的一半（小顶堆）——堆顶是较大半的最小值

**平衡不变量**：$|len(lo) - len(hi)| \le 1$。

$$\text{median} = \begin{cases} lo[0] \ (\text{取负还原}) & \text{总数为奇数} \\ \dfrac{lo[0] + hi[0]}{2} \ (\text{注意 lo 存了负号}) & \text{总数为偶数} \end{cases}$$

**复杂度**：插入 $O(\log n)$，查中位数 $O(1)$。

In [ ]:
class MedianFinder:
    def __init__(self):
        self.lo = []      # 大顶堆（存负值）：较小一半
        self.hi = []      # 小顶堆：较大一半

    def add_num(self, num):
        heapq.heappush(self.lo, -num)             # 先进较小半
        heapq.heappush(self.hi, -heapq.heappop(self.lo))  # 把较小半最大挪到较大半
        if len(self.lo) < len(self.hi):           # 保持 lo >= hi
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def find_median(self):
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])             # 奇数：较小半堆顶
        return (-self.lo[0] + self.hi[0]) / 2.0   # 偶数：两堆顶均值

mf = MedianFinder()
for x in [1, 2, 3]:
    mf.add_num(x)
print('流 [1,2,3] 中位数 =', mf.find_median())   # 期望 2.0
mf.add_num(4)
print('流 [1,2,3,4] 中位数 =', mf.find_median())  # 期望 2.5
print('复杂度：add O(log n)，查询 O(1)')

## §5 合并 K 个有序链表（LeetCode 23）

**题目**：合并 k 个有序链表为一个有序链表。

**堆解法**：把每个链表的**当前头**放进小顶堆，每次弹出最小并接上，再推进该链表：

**复杂度**：$O(N \log k)$（N 为总节点数），空间 $O(k)$。

**对比**：直接两两合并 $O(Nk)$；分治归并 $O(N \log k)$ 同堆。

> 💡 堆里不能直接存 `ListNode`（不可比较），要存 `(val, i, node)` 三元组加序号防比较冲突。

In [ ]:
def merge_k_lists(lists):
    hp = []
    for i, head in enumerate(lists):
        if head is not None:
            heapq.heappush(hp, (head.val, i, head))
    dummy = tail = ListNode(0)
    while hp:
        val, i, node = heapq.heappop(hp)
        tail.next = node
        tail = tail.next
        nxt = node.next
        if nxt is not None:
            heapq.heappush(hp, (nxt.val, i, nxt))
    return dummy.next

class ListNode:
    def __init__(self, val=0, nxt=None):
        self.val = val
        self.next = nxt

def to_list(lists):
    # 便利：[[1,4,5],[1,3,4],[2,6]] -> 链表数组
    res = []
    for a in lists:
        dummy = cur = ListNode(0)
        for x in a:
            cur.next = ListNode(x)
            cur = cur.next
        res.append(dummy.next)
    return res

merged = merge_k_lists(to_list([[1, 4, 5], [1, 3, 4], [2, 6]]))
vals = []
while merged:
    vals.append(merged.val)
    merged = merged.next
print('合并后:', vals)   # 期望 [1,1,2,3,4,4,5,6]
print('复杂度：O(N log k)')

## §6 堆排序（了解 + 手撕备选）

**过程**：建堆 → 反复把堆顶（最大/最小）交换到末尾并缩小堆：

$$\(1) \ \text{heapify}(O(n)) \quad (2) \ n-1 \ \text{次 pop}(O(n \log n)) \Rightarrow O(n \log n)$$

- 不稳定排序（相等元素相对顺序可能变）
- 原地排序空间 $O(1)$；Python 工程上用 `sorted()`（Timsort，稳定）

**面试点**：堆排序 vs 快排——堆排最坏 $O(n \log n)$ 但实际常数大且不稳定；快排平均更快，最坏 $O(n^2)$。

## §7 常见 bug 与边界（面试踩坑区）

- **大顶堆用负数**：`heapq` 只支持小顶堆，存 `-x` 取回时**别忘了取负还原**
- **K 与下标混淆**：第 K 大（1-based）≠ 第 len-k 小；`find_kth_largest` 堆大小要正好 k
- **heapreplace vs push+pop**：`heapreplace(hp, x)` = 弹顶 + 压入，比两次操作快；但要求 hp 非空
- **中位数堆失衡**：插入后没保持 `|len(lo)-len(hi)|<=1`，奇偶判错
- **ListNode 不可比较**：直接 `heappush(hp, node)` 会 TypeError（因为 `__lt__` 缺失）——存 `(val, i, node)`
- **heapify 复杂度误判**：面试答「建堆 O(n log n)」会扣分——是 $O(n)$
- **原地修改输入**：`heappop` 改变列表；要保留原数据就 copy

## §8 面试追问与扩展（加分项）

### 8.1 为什么 Top-K 用「大小为 k 的小顶堆」而不是「大顶堆」

要找**前 K 大**，堆里留最小元素当「门槛」：新元素比门槛大才替换；用大顶堆则需要把整个堆当「候选」，无法淘汰。

### 8.2 建堆为什么 O(n)

自底向上下沉：大多数节点在底层，下沉次数 = 高度，半层以上节点基本只下沉 1 次——总和收敛到 $O(n)$。

### 8.3 流式 Top-K / 中位数的工程意义

实时排行榜（直播热度）、监控指标、推荐漏斗——堆是应对「数据持续到来」的标准结构。

### 8.4 相关题

378 有序矩阵中第 K 小的元素、373 查找和最小的 K 对数字、264 丑数 II（堆/DP）

## §9 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 题）

- [ ] 1. 数组中的第 K 个最大元素（LeetCode 215）
- [ ] 2. 前 K 个高频元素（347）
- [ ] 3. 数据流中的第 K 大元素（703）
- [ ] 4. 最后一块石头的重量（1046）
- [ ] 5. 最小栈（155，堆对比）
- [ ] 6. 数组中的第 K 小的元素（快速选择/堆，面试手撕）
- [ ] 7. 查找和最小的 K 对数字（373）
- [ ] 8. 根据字符出现频率排序（451）
- [ ] 9. 前 K 个最小数（排序/堆双向题）
- [ ] 10. 有序矩阵中第 K 小的元素（378）

### L2 进阶（10 题）

- [ ] 11. 合并 K 个升序链表（23）
- [ ] 12. 数据流的中位数（295）
- [ ] 13. 滑动窗口最大值（239，堆/单调队列双解）
- [ ] 14. 出现次数最多的 K 个单词（692，堆 + 排序规则）
- [ ] 15. 丑数 II（264）
- [ ] 16. 会议室 II（253，堆解）
- [ ] 17. 任务调度器（621，贪心+堆）
- [ ] 18. 雇佣 K 名工人的最低成本（857，堆+贪心）
- [ ] 19. 最接近原点的 K 个点（973）
- [ ] 20. 最大堆手写（面试手撕）

### L3 挑战（5 题）

- [ ] 21. 堆排序手写（稳定版/不稳定版说明）
- [ ] 22. K 个一组翻转链表 + 堆混合（困难题）
- [ ] 23. IPO（502，双堆贪心）
- [ ] 24. 课程表 III（630）
- [ ] 25. 天际线问题（218，扫描线+堆）

### 自测要点

- 5 分钟内盲写：215 的堆解法 + 347
- 讲清 heapify 为什么 O(n)、Top-K 为什么用<小顶堆>
- 能手写 295 双堆的平衡逻辑

## 附录：参考与下节预告

- Python 官方文档 `heapq` 源码（`_siftup/_siftdown` 实现）
- 《算法导论》第 6 章：堆排序

### 下节预告：主题 12 — 位运算

- 异或性质、`x & (x-1)`、`x & -x`、DP 组合技巧

> 💡 本主题验收：`高频题单.md`「堆」一节全部打勾，能手写 push/pop 与解释建堆复杂度。